# Week 5 Assignment - Task 2
## Spark Fundamentals: Theory & Code Questions

Using the same dataset from Task 1 on Databricks.

### Loading the Dataset

In [0]:


from pyspark.sql.functions import col, count, sum, avg, min, max, when, trim
from pyspark.sql.types import TimestampType

df = spark.read.csv(
    "/Volumes/workspace/default/assigment5/asigment5.csv",
    header=True,
    inferSchema=True
)

df.show(5)
print(f"Total rows: {df.count()}")
df.printSchema()

+-------+----------------+-----------+-------+----------------+-----------+-----+------------+---+---------+--------+--------------------+---------+-------------------+
|user_id|transaction_date|       city| region|product_category|sale_amount|price|subscription|age| store_id|  status|               email| username|      raw_timestamp|
+-------+----------------+-----------+-------+----------------+-----------+-----+------------+---+---------+--------+--------------------+---------+-------------------+
|   1092|      2023-09-02|    Houston|  South|          Beauty|     520.22|13.14|  Enterprise| 20|STORE_004|    NULL|user1092@example.com|user_1092|2024-03-15 00:00:00|
|   1469|      2023-12-28|    Chicago|Midwest|           Books|      530.8| 38.5|  Enterprise| 27|STORE_003|    NULL|user1469@example.com|user_1469|2024-07-21 00:00:00|
|   1104|      2024-04-26|    Chicago|Midwest|     Electronics|     248.71|38.44|  Enterprise| 39|STORE_009|  Active|user1104@example.com|user_1104|2024-08

---
## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

**Answer:**

- MapReduce writes intermediate results to disk after every step which makes it really slow. Spark keeps everything in memory so its much faster.
- MapReduce only has Map and Reduce operations. For anything complex you have to chain multiple jobs. Spark gives us filter, join, groupBy etc out of the box.
- MapReduce cant do real time processing, its batch only. Spark supports both batch and streaming.
- Writing MapReduce in Java needs a lot of boilerplate code. PySpark is way simpler.
- For iterative stuff like ML algorithms, MapReduce reads/writes to disk every iteration. Spark caches data in memory so its up to 100x faster.

---
## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

**Answer:**

In MapReduce, every iteration does: read from disk → process → write back to disk. So if gradient descent runs 100 iterations, thats 100 read/write cycles to HDFS which is very slow.

Spark solves this by loading data into memory once and keeping it there using `.cache()` or `.persist()`. All iterations read from memory directly — no disk I/O in between. Only the final result gets written to disk.

```
MapReduce: Disk → RAM → Process → Disk → RAM → Process → Disk (every iteration)
Spark:     Disk → RAM → Process → Process → Process → Disk (once at end)
```

Thats why Spark is up to 100x faster for ML workloads.

---
## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [0]:
print(f"Before removing duplicates: {df.count()} rows")

# Removing duplicates based on user_id and transaction_date
df_no_duplicates = df.dropDuplicates(["user_id", "transaction_date"])

print(f"After removing duplicates: {df_no_duplicates.count()} rows")
print(f"Duplicates removed: {df.count() - df_no_duplicates.count()}")

Before removing duplicates: 1000 rows
After removing duplicates: 970 rows
Duplicates removed: 30


In [0]:
#verifying no duplicates remain
df_no_duplicates.groupBy("user_id", "transaction_date") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-------+----------------+-----+
|user_id|transaction_date|count|
+-------+----------------+-----+
+-------+----------------+-----+



---
## Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [0]:
df_sales = df

#filter west region and get avg sale_amount per category
west_avg_sales = df_sales.filter(col("region") == "West") \
    .groupBy("product_category") \
    .agg(avg("sale_amount").alias("avg_sale_amount")) \
    .orderBy(col("avg_sale_amount").desc())

west_avg_sales.show()

+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|     Electronics| 535.6330952380953|
|          Beauty|            523.62|
|            Home| 513.7823999999999|
|           Books| 512.1033962264153|
|            Food| 509.3487500000001|
|        Clothing|  505.279534883721|
|          Sports|503.58682926829283|
|            Toys| 423.1641860465116|
+----------------+------------------+



---
## Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

**Answer:**

- `.na.drop()` removes rows that have null values. It reduces the total row count.
- `.na.fill()` replaces nulls with a value we specify. Row count stays the same.

We use `drop` when null rows are bad/useless data. We use `fill` when we want to keep the rows but just replace the missing values with something like a default. In our dataset status had 404 nulls so dropping would lose too much data, filling with "Unknown" makes more sense.

In [0]:
#checking nulls before
null_count_before = df.filter(col("status").isNull()).count()
print(f"Null values in status before fill: {null_count_before}")

#filling null status with 'Unknown'
df_filled = df.na.fill({"status": "Unknown"})

#verify
null_count_after = df_filled.filter(col("status").isNull()).count()
print(f"Null values in status after fill: {null_count_after}")

df_filled.filter(col("status") == "Unknown").show(5)

Null values in status before fill: 404
Null values in status after fill: 0
+-------+----------------+-----------+-------+----------------+-----------+------+------------+---+---------+-------+--------------------+---------+-------------------+
|user_id|transaction_date|       city| region|product_category|sale_amount| price|subscription|age| store_id| status|               email| username|      raw_timestamp|
+-------+----------------+-----------+-------+----------------+-----------+------+------------+---+---------+-------+--------------------+---------+-------------------+
|   1092|      2023-09-02|    Houston|  South|          Beauty|     520.22| 13.14|  Enterprise| 20|STORE_004|Unknown|user1092@example.com|user_1092|2024-03-15 00:00:00|
|   1469|      2023-12-28|    Chicago|Midwest|           Books|      530.8|  38.5|  Enterprise| 27|STORE_003|Unknown|user1469@example.com|user_1469|2024-07-21 00:00:00|
|   1438|      2024-02-14|    Chicago|Midwest|            Toys|     799.32|186.8

---
## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [0]:
#count per city, only where count > 100
city_counts = df.groupBy("city") \
    .agg(count("*").alias("total_records")) \
    .filter(col("total_records") > 100) \
    .orderBy(col("total_records").desc())

city_counts.show()

+-----------+-------------+
|       city|total_records|
+-----------+-------------+
|   New York|          120|
|Los Angeles|          113|
|    Chicago|          111|
|    Houston|          109|
|    Phoenix|          108|
+-----------+-------------+



In [0]:
#all cities for reference
df.groupBy("city") \
    .agg(count("*").alias("total_records")) \
    .orderBy(col("total_records").desc()) \
    .show()

+-------------+-------------+
|         city|total_records|
+-------------+-------------+
|     New York|          120|
|  Los Angeles|          113|
|      Chicago|          111|
|      Houston|          109|
|      Phoenix|          108|
| Philadelphia|           94|
|  San Antonio|           87|
|    San Diego|           83|
|       Dallas|           80|
|     San Jose|           79|
|      Memphis|            7|
|   Louisville|            4|
|    Las Vegas|            3|
|     Portland|            1|
|Oklahoma City|            1|
+-------------+-------------+



---
## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

**Answer:**

Spark DataFrames are immutable meaning once created they cant be changed. Every operation like drop, rename, filter etc creates a new DataFrame instead of modifying the original.

So when cleaning data we have to reassign the result:
```python
df.drop("email")  # this does nothing to df
df = df.drop("email")  # this captures the change
```

The good thing is the original data is always safe. If a cleaning step goes wrong we can just go back to the original df. Also immutability is what makes Spark fault tolerant — if a partition is lost it can recompute from the lineage.

In [0]:
#demonstrating immutability
print(f"Original columns: {df.columns}")

df_dropped = df.drop("raw_timestamp")

print(f"Original still has raw_timestamp: {'raw_timestamp' in df.columns}")
print(f"New df has raw_timestamp: {'raw_timestamp' in df_dropped.columns}")

Original columns: ['user_id', 'transaction_date', 'city', 'region', 'product_category', 'sale_amount', 'price', 'subscription', 'age', 'store_id', 'status', 'email', 'username', 'raw_timestamp']
Original still has raw_timestamp: True
New df has raw_timestamp: False


---
## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [0]:
#filter age 18-30 and premium subscription
filtered_df = df.filter(
    (col("age").between(18, 30)) & 
    (col("subscription") == "Premium")
)

print(f"Total matching records: {filtered_df.count()}")
filtered_df.select("user_id", "age", "subscription", "city", "product_category", "sale_amount").show(10)

Total matching records: 67
+-------+---+------------+-----------+----------------+-----------+
|user_id|age|subscription|       city|product_category|sale_amount|
+-------+---+------------+-----------+----------------+-----------+
|   1012| 24|     Premium|    Houston|            Toys|     389.03|
|   1486| 20|     Premium|Los Angeles|            Food|     530.44|
|   1629| 24|     Premium|    Phoenix|     Electronics|      894.2|
|   1630| 28|     Premium|     Dallas|     Electronics|     907.96|
|   1440| 22|     Premium|    Chicago|        Clothing|     584.71|
|   1523| 19|     Premium|     Dallas|           Books|     769.44|
|   1757| 28|     Premium|    Houston|           Books|     455.07|
|   1366| 21|     Premium|     Dallas|            Toys|     483.21|
|   1719| 27|     Premium|    Chicago|            Food|     461.88|
|   1446| 22|     Premium|Los Angeles|            Toys|       NULL|
+-------+---+------------+-----------+----------------+-----------+
only showing top 10 r

---
## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

**Answer:**

Spark skips null values in aggregation functions by default. So `avg(sale_amount)` only averages non-null rows which can give misleading results if there are many nulls.

Also `count("*")` counts all rows but `count("sale_amount")` only counts non-null ones. If we dont handle nulls first, our counts wont match up and ratios will be wrong.

So its better to decide upfront — fill nulls with 0, fill with the average, or drop those rows — then do aggregations on clean data.

In [0]:
#showing how nulls affect aggregations
print("Without handling nulls:")
df.select(
    count("*").alias("total_rows"),
    count("sale_amount").alias("non_null_sales"),
    avg("sale_amount").alias("avg_sale")
).show()

print("After filling nulls with 0:")
df.fillna({"sale_amount": 0}).select(
    count("*").alias("total_rows"),
    count("sale_amount").alias("non_null_sales"),
    avg("sale_amount").alias("avg_sale")
).show()
#notice the average changes because 0s pull it down

Without handling nulls:
+----------+--------------+-----------------+
|total_rows|non_null_sales|         avg_sale|
+----------+--------------+-----------------+
|      1000|           949|500.4175869336145|
+----------+--------------+-----------------+

After filling nulls with 0:
+----------+--------------+------------------+
|total_rows|non_null_sales|          avg_sale|
+----------+--------------+------------------+
|      1000|          1000|474.89629000000014|
+----------+--------------+------------------+



---
## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [0]:
from pyspark.sql.types import TimestampType

#casting raw_timestamp to TimestampType and renaming
df_with_event_time = df \
    .withColumn("raw_timestamp", col("raw_timestamp").cast(TimestampType())) \
    .withColumnRenamed("raw_timestamp", "event_time")

df_with_event_time.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- subscription: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import to_timestamp, col

df_with_event_time = df.withColumn(
    "event_time",
    to_timestamp(col("raw_timestamp"), "dd/MM/yyyy HH:mm")
)


---
## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

**Answer:**

When we do `groupBy("region")`, Spark needs all records with the same region value on the same partition to compute the aggregate. But the data is spread randomly across partitions. So Spark has to redistribute/move the data across the network — this is called a **shuffle**.

Its a **wide transformation** because each output partition needs data from multiple input partitions. Unlike narrow transformations (like filter) where each partition works independently, wide transformations require data to move between nodes.

Shuffles are expensive because they involve serializing data, writing to disk, sending over the network, and deserializing. Thats why we should try to minimize shuffles when possible.

In [0]:
#we can see the shuffle in the execution plan
region_grouped = df.groupBy("region").agg(count("*").alias("total"))
region_grouped.explain()
#the Exchange in the plan shows the shuffle happening

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[region#20770], functions=[finalmerge_count(merge count#20786L) AS count(1)#20782L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#19411]
               +- PhotonShuffleExchangeSink hashpartitioning(region#20770, 16)
                  +- PhotonGroupingAgg(keys=[region#20770], functions=[partial_count(1) AS count#20786L])
                     +- PhotonRowToColumnar
                        +- FileScan csv [region#20770] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/assigment5/asigment5.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<region:string>


== Photon Explanation ==
The query is fully supported by Photon.


---
## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [0]:
#checking how many bad rows exist
bad_rows = df.filter(
    (col("email").isNull()) | (col("username") == "")
)
print(f"Bad rows found: {bad_rows.count()}")
bad_rows.select("user_id", "email", "username").show(10, truncate=False)

Bad rows found: 28
+-------+-----+---------+
|user_id|email|username |
+-------+-----+---------+
|1106   |NULL |user_1106|
|1124   |NULL |user_1124|
|1199   |NULL |user_1199|
|1166   |NULL |user_1166|
|1626   |NULL |user_1626|
|1866   |NULL |user_1866|
|1607   |NULL |user_1607|
|1726   |NULL |user_1726|
|1503   |NULL |user_1503|
|1014   |NULL |user_1014|
+-------+-----+---------+
only showing top 10 rows


In [0]:
#removing those rows using NOT condition
df_cleaned = df.filter(
    ~(
        (col("email").isNull()) | (col("username") == "")
    )
)

print(f"Before: {df.count()} rows")
print(f"After: {df_cleaned.count()} rows")
print(f"Removed: {df.count() - df_cleaned.count()}")

Before: 1000 rows
After: 938 rows
Removed: 62


---
## Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

**Answer:**

`.agg()` lets us apply multiple aggregation functions in one go. We just pass multiple expressions separated by commas. It works both with and without `groupBy`.

In [0]:
#overall stats for price column
df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    avg("price").alias("mean_price")
).show()

+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|     5.69|   498.17|250.64428872497314|
+---------+---------+------------------+



In [0]:
#with groupBy — per category stats
df.groupBy("product_category").agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    avg("price").alias("mean_price"),
    count("price").alias("total_records")
).orderBy("product_category").show()

+----------------+---------+---------+------------------+-------------+
|product_category|min_price|max_price|        mean_price|total_records|
+----------------+---------+---------+------------------+-------------+
|          Beauty|     8.35|   494.69| 245.0332773109244|          119|
|           Books|     17.0|   496.71| 273.6482677165356|          127|
|        Clothing|     7.31|   493.51|248.37618644067783|          118|
|     Electronics|     7.35|   498.17|240.69786324786315|          117|
|            Food|     5.69|   477.44|244.60533333333336|          120|
|            Home|     6.32|   496.03|243.75125000000014|          120|
|          Sports|     9.65|   497.54|249.96918181818177|          110|
|            Toys|     5.78|   494.25|257.45508474576286|          118|
+----------------+---------+---------+------------------+-------------+



---
## Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

**Answer:**

`inferSchema=True` makes Spark guess column types by sampling the data. The problem is when dates have mixed formats like `2024-03-15` and `19/05/2023`, Spark either:
- Reads the whole column as string (cant do date operations on it)
- Or infers it as timestamp and silently sets inconsistent values to null

In our dataset `raw_timestamp` has formats like `yyyy-MM-dd HH:mm:ss`, `dd/MM/yyyy HH:mm`, and `yyyy-MM-ddTHH:mm:ssZ`. Spark inferred it as string because of this mix. If we try casting it to timestamp, the dd/MM/yyyy ones become null without any error.

Better approach is to define schema explicitly and parse dates manually using `to_timestamp()` with the correct format.

In [0]:
#showing the mixed formats in raw_timestamp
print("Data type:", dict(df.dtypes)["raw_timestamp"])
df.select("raw_timestamp").distinct().show(10, truncate=False)

Data type: string
+--------------------+
|raw_timestamp       |
+--------------------+
|2024-12-27T00:00:00Z|
|2024-04-06 00:00:00 |
|2023-11-01 00:00:00 |
|2023-09-06 00:00:00 |
|2024-10-10 00:00:00 |
|2024-12-21 00:00:00 |
|2024-07-14 00:00:00 |
|2024-02-15 00:00:00 |
|11/12/2024 00:00    |
|2024-01-20 00:00:00 |
+--------------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import col, expr
df_cast = df.withColumn("parsed_ts", expr("try_cast(raw_timestamp as timestamp)"))

original_non_null = df.filter(col("raw_timestamp").isNotNull()).count()
parsed_non_null = df_cast.filter(col("parsed_ts").isNotNull()).count()

print(f"Non-null raw values: {original_non_null}")
print(f"Successfully parsed: {parsed_non_null}")
print(f"Lost due to bad format: {original_non_null - parsed_non_null}")


Non-null raw values: 1000
Successfully parsed: 898
Lost due to bad format: 102


---
## Q15: Write a final processing pipeline that:
1. Filters out duplicates
2. Fills null prices with 0
3. Groups by store_id to calculate total revenue

In [0]:
from pyspark.sql import functions as F

pipeline_result = (
    df
    # step 1: remove duplicates
    .dropDuplicates()
    # step 2: fill null prices with 0
    .fillna({"price": 0, "sale_amount": 0})
    # step 3: group by store_id and calculate total revenue
    .groupBy("store_id")
    .agg(
        F.round(F.sum("sale_amount"), 2).alias("total_revenue"),
        F.count("*").alias("total_transactions"),
        F.round(F.avg("sale_amount"), 2).alias("avg_revenue_per_txn")
    )
    .orderBy(F.col("total_revenue").desc())
)

pipeline_result.show()

+---------+-------------+------------------+-------------------+
| store_id|total_revenue|total_transactions|avg_revenue_per_txn|
+---------+-------------+------------------+-------------------+
|STORE_013|     32506.29|                62|              524.3|
|STORE_011|     28441.53|                53|             536.63|
|STORE_014|     26997.01|                53|             509.38|
|STORE_009|     26906.69|                57|             472.05|
|STORE_003|     26625.14|                57|             467.11|
|STORE_020|     25107.59|                50|             502.15|
|STORE_005|     24398.64|                54|             451.83|
|STORE_002|     24315.15|                50|              486.3|
|STORE_010|     24007.95|                56|             428.71|
|STORE_016|     23154.68|                48|             482.39|
|STORE_019|     21877.59|                52|             420.72|
|STORE_007|     21501.97|                45|             477.82|
|STORE_017|     21302.77|